# 인버터별 sn구분하여 모델 생성코드

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import re
import joblib

# 데이터 로드
solar_data_path = '/content/drive/MyDrive/태양광데이터/태양광데이터_24-03-21.csv'
weather_data_path = '/content/drive/MyDrive/태양광데이터/날씨데이터_24-03-21.csv'

solar_df = pd.read_csv(solar_data_path)
weather_df = pd.read_csv(weather_data_path)

# 태양광 데이터 전처리
solar_df['crt_dttm'] = pd.to_datetime(solar_df['crt_dttm'], errors='coerce') # object를 datetime64[ns]로, coerce: 변환 실패 → NaT
solar_df['datetime'] = solar_df['crt_dttm'].dt.floor('h') # 1시간 단위로 내림
solar_df['date'] = solar_df['crt_dttm'].dt.date # 년월일, 2022-12-26
solar_df['hour'] = solar_df['crt_dttm'].dt.hour # 시각
solar_df['month'] = solar_df['crt_dttm'].dt.month # 월
solar_df['day'] = solar_df['crt_dttm'].dt.day # 일

# 각 시간별 마지막 데이터 선택
hourly_solar = solar_df.sort_values(by=['crt_dttm']).groupby(['PowerPlantID', 'SN', 'datetime']).last().reset_index() # 그룹의 마지막 행 선택, first(): 그룹의 첫 행 선택

# 각 시간별 발전량 계산
hourly_solar['date'] = hourly_solar['datetime'].dt.date  # 날짜 열 추가, 중복 코드로 보임 solar_df에서 date 열 생성함.
hourly_solar['DYield_diff'] = hourly_solar.groupby(['PowerPlantID', 'SN', 'date'])['DYield'].diff().fillna(hourly_solar['DYield']) # .diff(): 이전 값과의 차이

# 날씨 데이터 전처리
weather_df['base_time'] = weather_df['base_time'].apply(lambda x: f"{int(x):04d}") # 4자리 고정 및 obj, 0 -> '0000'
weather_df['datetime'] = pd.to_datetime(weather_df['base_date'].astype(str) + weather_df['base_time'], format='%Y%m%d%H%M') # 2023-06-27 10:00:00
weather_df['datetime'] = weather_df['datetime'].dt.floor('h') # 1시간 단위로 내림

# 필요한 컬럼 선택
solar_selected = hourly_solar[['PowerPlantID', 'SN', 'datetime', 'DYield_diff', 'crt_dttm']]
weather_selected = weather_df[['solarpower_sn', 'datetime', 'rn1', 'sky', 't1h', 'reh', 'rn1v', 'wsd']]

# 시간 차이를 두고 데이터 병합
merged_df = pd.merge(solar_selected, weather_selected, left_on=['PowerPlantID', 'datetime'], right_on=['solarpower_sn', 'datetime'], how='inner') # PowerPlantID랑 solarpower_sn랑 같은 키로 간주하여 innerjoin
merged_df['datetime'] = merged_df['datetime'] + pd.Timedelta(hours=1) # .last() 데이터들이 대부분 55분짜리가 많은 그래서 1시간 뒤 데이터로 재해석

# rn1v에서 \N을 NaN으로 처리
merged_df['rn1v'] = merged_df['rn1v'].replace('\\N', np.nan).astype(float) # \N을 NaN으로 변경 후 type float으로 변경
merged_df['rn1v'].fillna(0, inplace=True) # NaN값 0으로 변경 inplace로

# 강수량 변환 함수
def convert_pcp(value):
    if '강수없음' in value:
        return 0.0
    elif '미만' in value:
        num = re.findall(r'\d+', value)
        if num:
            return float(num[0]) - 0.5 # 상한값보다 조금작은값으로 변환 1 -> 0.5, 3 -> 2.5
        else:
            return 0.0
    else:
        num = re.findall(r'\d+\.\d+|\d+', value) # 매칭되는 문자열을 전부 찾아서 list로 반환
        if num:
            return float(num[0]) # num의 type이 list
        else:
            return 0.0

merged_df['rn1'] = merged_df['rn1'].apply(convert_pcp)

# 피처 엔지니어링
merged_df['hour'] = merged_df['datetime'].dt.hour
merged_df['month'] = merged_df['datetime'].dt.month
merged_df['day'] = merged_df['datetime'].dt.day

# 인버터별로 데이터 나누기
inverters = {
    'YAG_HSM_SWR_0001': [1015642044025, 1015642044102, 1015642044093],
    'GJM_GSG_SGD_0001': [1018282223012, 1018282225009]
}

results_dict = {}

for plant_id, sn_list in inverters.items():
    for sn in sn_list:
        inverter_data = merged_df[(merged_df['PowerPlantID'] == plant_id) & (merged_df['SN'] == sn)].reset_index(drop=True)

        if len(inverter_data) < 2:  # 샘플 수가 너무 적으면 다음 인버터로 넘어감
            print(f"Not enough data for PowerPlantID: {plant_id}, SN: {sn}. Skipping this inverter. Sample: {len(inverter_data)}")
            continue

        # 데이터 확인
        print(f'Inverter data for PowerPlantID: {plant_id}, SN: {sn}')
        print(inverter_data.head())

        # 필요한 피처 선택
        X = inverter_data[['rn1', 'sky', 't1h', 'reh', 'rn1v', 'wsd', 'hour', 'month', 'day']]
        y = inverter_data['DYield_diff']

        # 데이터 분리
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 모델 학습
        model = RandomForestRegressor(max_depth=20, max_features='sqrt', n_estimators=1500, random_state=42, verbose=1)
        model.fit(X_train, y_train)

        # 모델 저장
        model_filename = f'/content/drive/MyDrive/태양광데이터/model_{plant_id}_{sn}.joblib'
        joblib.dump(model, model_filename)

        # 예측
        y_pred = model.predict(X_test)

         # 평가
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # 실제 값이 0이 아닌 데이터만 사용하여 MAPE 계산
        non_zero_actuals = y_test != 0
        mape = np.mean(np.abs((y_test[non_zero_actuals] - y_pred[non_zero_actuals]) / y_test[non_zero_actuals])) * 100

        mbd = np.mean(y_pred - y_test)
        print(f'PowerPlantID: {plant_id}, SN: {sn}')
        print(f'Mean Squared Error: {mse}')
        print(f'Root Mean Squared Error: {rmse}')
        print(f'Mean Absolute Error: {mae}')
        print(f'R-squared: {r2}')
        print(f'Mean Absolute Percentage Error: {mape}%')
        print(f'Mean Bias Deviation: {mbd}')

        # 실제값과 예측값 데이터프레임 생성
        results_df = pd.DataFrame({
            'DateTime': inverter_data.loc[X_test.index, 'datetime'].values,
            'PowerPlantID': inverter_data.loc[X_test.index, 'PowerPlantID'].values,
            'SN': inverter_data.loc[X_test.index, 'SN'].values,
            'crt_dttm': inverter_data.loc[X_test.index, 'crt_dttm'].values,
            'Actual': y_test.values,
            'Predicted': y_pred
        }).reset_index(drop=True)

        results_dict[f'{plant_id}_{sn}'] = results_df

        # 데이터프레임 정렬
        results_df = results_df.sort_values(by='DateTime').reset_index(drop=True)

        # 시각화
        plt.figure(figsize=(14, 7))
        plt.plot(results_df['DateTime'], results_df['Actual'], label='Actual', marker='o')
        plt.plot(results_df['DateTime'], results_df['Predicted'], label='Predicted', marker='o')
        plt.legend()
        plt.xlabel('DateTime')
        plt.ylabel('Hourly Yield')
        plt.title(f'Actual vs Predicted Hourly Yield (PowerPlantID: {plant_id}, SN: {sn})')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        # DateTime에서 날짜만 추출하는 컬럼 추가
        results_df['Date'] = results_df['DateTime'].dt.date

        # 각 날짜별로 마지막 시간 데이터를 추출
        last_results_df = results_df.sort_values(by=['DateTime']).groupby(['PowerPlantID', 'SN', 'Date']).last().reset_index()

        # 불필요한 Date 컬럼 제거
        last_results_df = last_results_df.drop(columns=['Date'])
        last_results_df = last_results_df.sort_values(by='DateTime').reset_index(drop=True)

        # 실제 값이 0이 아닌 데이터만 사용하여 MAPE 계산
        non_zero_actuals = last_results_df[last_results_df['Actual'] != 0]

        # 평가
        mse = mean_squared_error(last_results_df['Actual'], last_results_df['Predicted'])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(last_results_df['Actual'], last_results_df['Predicted'])
        r2 = r2_score(last_results_df['Actual'], last_results_df['Predicted'])
        mape = np.mean(np.abs((non_zero_actuals['Actual'] - non_zero_actuals['Predicted']) / non_zero_actuals['Actual'])) * 100
        mbd = np.mean(last_results_df['Predicted'] - last_results_df['Actual'])
        print(f'Mean Squared Error (Daily): {mse}')
        print(f'Root Mean Squared Error (Daily): {rmse}')
        print(f'Mean Absolute Error (Daily): {mae}')
        print(f'R-squared (Daily): {r2}')
        print(f'Mean Absolute Percentage Error (Daily): {mape}%')
        print(f'Mean Bias Deviation (Daily): {mbd}')

        # 시각화 (Daily)
        plt.figure(figsize=(14, 7))
        plt.plot(last_results_df['DateTime'], last_results_df['Actual'], label='Actual', marker='o')
        plt.plot(last_results_df['DateTime'], last_results_df['Predicted'], label='Predicted', marker='o')
        plt.legend()
        plt.xlabel('DateTime')
        plt.ylabel('Hourly Yield')
        plt.title(f'Daily Actual vs Predicted Hourly Yield (PowerPlantID: {plant_id}, SN: {sn})')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# 각 인버터별로 저장된 결과를 확인할 수 있습니다.
for key, df in results_dict.items():
    df['Difference'] = df['Actual'] - df['Predicted']
    df['Accuracy'] = np.where(df['Actual'] == 0, 0, (1 - abs(df['Difference']) / abs(df['Actual'])) * 100)
    print(f"Results for {key}:")
    print(df)

    # # 인버터별 결과를 개별 파일로 저장
    # df.to_csv(f'results_{key}.csv', index=False)

# 태양광 발전량 인버터별로 나누어 예측하고 실제값이랑 비교하는 코드

In [ ]:
import pandas as pd
import numpy as np
import joblib
import re
import matplotlib.pyplot as plt

# 새로운 날씨 데이터 로드
new_weather_data_path = '/content/drive/MyDrive/태양광데이터/날씨데이터_24-07-29-11시.csv'
new_weather_df = pd.read_csv(new_weather_data_path)

# 날씨 데이터 전처리
new_weather_df['base_time'] = new_weather_df['base_time'].apply(lambda x: f"{int(x):04d}")
new_weather_df['datetime'] = pd.to_datetime(new_weather_df['base_date'].astype(str) + new_weather_df['base_time'], format='%Y%m%d%H%M')
new_weather_df['datetime'] = new_weather_df['datetime'].dt.floor('H')
new_weather_df['datetime'] = new_weather_df['datetime'] + pd.Timedelta(hours=1)

# 필요한 컬럼 선택
weather_selected = new_weather_df[['solarpower_sn', 'datetime', 'rn1', 'sky', 't1h', 'reh', 'rn1v', 'wsd']]

# \N을 NaN으로 대체하고 float로 변환
weather_selected.loc[:, 'rn1v'] = weather_selected['rn1v'].replace('\\N', np.nan).astype(float)
weather_selected.loc[:, 'rn1v'] = weather_selected['rn1v'].fillna(0)
weather_selected.loc[:, 'sky'] = weather_selected['sky'].replace('\\N', np.nan).astype(float)
weather_selected.loc[:, 'sky'] = weather_selected['sky'].fillna(0)
# 강수량 변환 함수
def convert_pcp(value):
    if '강수없음' in value:
        return 0.0
    elif '미만' in value:
        num = re.findall(r'\d+', value)
        if num:
            return float(num[0]) - 0.5
        else:
            return 0.0
    else:
        num = re.findall(r'\d+\.\d+|\d+', value)
        if num:
            return float(num[0])
        else:
            return 0.0

weather_selected.loc[:, 'rn1'] = weather_selected['rn1'].apply(convert_pcp)

# 피처 엔지니어링
weather_selected.loc[:, 'hour'] = weather_selected['datetime'].dt.hour
weather_selected.loc[:, 'month'] = weather_selected['datetime'].dt.month
weather_selected.loc[:, 'day'] = weather_selected['datetime'].dt.day


# 특정 일자 선택 (예: 2024-07-23부터 2024-07-29까지)
start_date = '2024-07-23'
end_date = '2024-07-29'
mask = (weather_selected['datetime'] >= start_date) & (weather_selected['datetime'] <= end_date)
selected_weather_data = weather_selected[mask]

# 인버터별 모델 로드 및 예측
inverters = {
    'YAG_HSM_SWR_0001': [1015642044025, 1015642044102, 1015642044093],
    'GJM_GSG_SGD_0001': [1018282223012, 1018282225009]
}

prediction_results = []

for plant_id, sn_list in inverters.items():
    for sn in sn_list:
        model_filename = f'/content/drive/MyDrive/태양광데이터/model_{plant_id}_{sn}.joblib'
        try:
            model = joblib.load(model_filename)
        except FileNotFoundError:
            print(f"Model for {plant_id}, SN: {sn} not found. Skipping this inverter.")
            continue

        # 해당 인버터의 날씨 데이터 선택
        inverter_weather_data = selected_weather_data [selected_weather_data ['solarpower_sn'] == plant_id]

        # 필요한 피처 선택
        X = inverter_weather_data[['rn1', 'sky', 't1h', 'reh', 'rn1v', 'wsd', 'hour', 'month', 'day']]

        # 예측
        predictions = model.predict(X)

        # 예측 결과 저장
        prediction_df = pd.DataFrame({
            'DateTime': inverter_weather_data['datetime'],
            'Prediction_Time': inverter_weather_data['datetime'] + pd.Timedelta(hours=1),
            'PowerPlantID': plant_id,
            'SN': sn,
            'rn1': inverter_weather_data['rn1'],
            'sky': inverter_weather_data['sky'],
            't1h': inverter_weather_data['t1h'],
            'reh': inverter_weather_data['reh'],
            'rn1v': inverter_weather_data['rn1v'],
            'wsd': inverter_weather_data['wsd'],
            'hour': inverter_weather_data['hour'],
            'month': inverter_weather_data['month'],
            'day': inverter_weather_data['day'],
            'Predicted_DYield': predictions
        })

        prediction_results.append(prediction_df)

# 결과를 하나의 데이터프레임으로 결합
all_predictions_df = pd.concat(prediction_results, ignore_index=True)

# 실제 태양광 데이터 로드
actual_solar_data_path = '/content/drive/MyDrive/태양광데이터/태양광데이터_24-07-29-11시.csv'
actual_solar_df = pd.read_csv(actual_solar_data_path)

# datetime 형식으로 변환
actual_solar_df['datetime'] = pd.to_datetime(actual_solar_df['crt_dttm'])

# 데이터프레임을 PowerPlantID, SN, datetime 기준으로 정렬
actual_solar_df = actual_solar_df.sort_values(['PowerPlantID', 'SN', 'datetime'])

# 각 시간 구간의 마지막 데이터 선택
actual_solar_df['datetime_floor'] = actual_solar_df['datetime'].dt.floor('H')
last_entries = actual_solar_df.groupby(['PowerPlantID', 'SN', 'datetime_floor']).tail(1)

# 필요한 시간대의 데이터만 선택
last_entries['Prediction_Time'] = last_entries['datetime_floor'] + pd.Timedelta(hours=1)

# 첫 번째 데이터는 첫 기록으로 그대로 사용
last_entries['DYield_diff'] = last_entries['DYield']

# 날짜별로 첫 번째 데이터를 구분
last_entries['date'] = last_entries['datetime'].dt.date

# 각 날짜별 첫 번째 시간의 발전량 차이는 그대로 두고, 그 외의 시간 구간은 차이를 계산
last_entries['DYield_diff'] = last_entries.groupby(['PowerPlantID', 'SN', 'date'])['DYield'].diff().fillna(last_entries['DYield'])

# 필요한 열만 선택
final_entries = last_entries[['PowerPlantID', 'SN', 'crt_dttm', 'datetime', 'DYield', 'DYield_diff', 'Prediction_Time']]

final_entries.reset_index(drop=True, inplace=True)

# 데이터 유형 일치시키기
all_predictions_df['PowerPlantID'] = all_predictions_df['PowerPlantID'].astype(str)
final_entries['PowerPlantID'] = final_entries['PowerPlantID'].astype(str)
all_predictions_df['SN'] = all_predictions_df['SN'].astype(str)
final_entries['SN'] = final_entries['SN'].astype(str)

# all_predictions_df와 final_entries 병합
comparison_df = pd.merge(all_predictions_df, final_entries,
                         left_on=['Prediction_Time', 'PowerPlantID', 'SN'],
                         right_on=['Prediction_Time', 'PowerPlantID', 'SN'],
                         suffixes=('_predicted', '_actual'))

# 필요한 컬럼 선택
comparison_df = comparison_df[['DateTime', 'Prediction_Time', 'PowerPlantID', 'SN',
                               'Predicted_DYield', 'DYield_diff']]

# 컬럼 이름 변경
comparison_df.columns = ['DateTime', 'Prediction_Time', 'PowerPlantID', 'SN',
                         'Predicted_DYield', 'Actual_DYield']

# 절대 오차 (Absolute Error) 계산
comparison_df['Absolute_Error'] = (comparison_df['Actual_DYield'] - comparison_df['Predicted_DYield']).abs()

# 상대 오차 (Relative Error) 계산
comparison_df['Relative_Error'] = comparison_df['Absolute_Error'] / comparison_df['Actual_DYield']

# 정확도 (Accuracy) 계산: 1 - 상대 오차
comparison_df['Accuracy'] = (1 - comparison_df['Relative_Error']) * 100

# 월별로 그룹화하여 그래프 생성
comparison_df['Month'] = comparison_df['Prediction_Time'].dt.month

months = comparison_df['Month'].unique()

for month in months:
    monthly_data = comparison_df[comparison_df['Month'] == month]
    power_plants = monthly_data['PowerPlantID'].unique()

    for plant in power_plants:
        plant_data = monthly_data[monthly_data['PowerPlantID'] == plant]
        sns = plant_data['SN'].unique()

        for sn in sns:
            sn_data = plant_data[plant_data['SN'] == sn]

            plt.figure(figsize=(14, 7))
            plt.plot(sn_data['Prediction_Time'], sn_data['Predicted_DYield'], label='Predicted DYield', marker='o')
            plt.plot(sn_data['Prediction_Time'], sn_data['Actual_DYield'], label='Actual DYield', marker='x')

            plt.xlabel('Prediction Time')
            plt.ylabel('DYield')
            plt.title(f'Comparison of Predicted and Actual DYield for Month {month} (PlantID: {plant}, SN: {sn})')
            plt.legend()
            plt.grid(True)
            plt.show()


# 예측값과, 실제값 데이터프레임 저장코드

In [ ]:
comparison_df.to_csv('/content/drive/MyDrive/태양광데이터/7-23~7-29일 태양광 발전량비교.csv', index=False)